# Synthetic Data Generator using Open-Source Tools

## Overview
This notebook demonstrates how to build a production-ready synthetic data generator using:
- **Faker**: For realistic structured data (names, emails, etc.)
- **Ollama**: For AI-generated contextual content using local LLMs
- **Pandas**: For data manipulation and export

## Use Case
Generate realistic employee datasets with AI-powered biographies that match job titles and experience levels.

## Step 1: Setup & Imports

First, we'll install the required libraries and import them into our notebook.

In [ ]:
# Install required packages
!pip install faker ollama pandas tqdm -q 

In [ ]:
# Import libraries
import pandas as pd
from faker import Faker
import ollama
from tqdm import tqdm
import random
from typing import Dict, List, Any
import warnings

warnings.filterwarnings('ignore')

print("✓ All libraries imported successfully!")

## Step 2: Configuration

Define the dataset schema and generation parameters in one centralized location for easy modification.

In [ ]:
# Configuration Dictionary
CONFIG = {
    # Generation Parameters
    "NUM_ROWS": 50,
    "BATCH_SIZE": 10,  # Process in batches to manage memory
    
    # Schema Definition
    "SCHEMA": {
        "Name": "faker",           # Generated by Faker
        "Email": "faker",          # Generated by Faker
        "Job Title": "faker",      # Generated by Faker
        "Seniority": "custom",     # Custom logic (Junior, Mid, Senior)
        "Years of Experience": "custom",  # Custom logic (based on seniority)
        "AI Generated Bio": "llm"  # Generated by Ollama LLM
    },
    
    # LLM Configuration
    "LLM_MODEL": "llama3",  # Options: llama3, mistral, phi, etc.
    "LLM_TEMPERATURE": 0.7,
    
    # Job Title Options
    "JOB_TITLES": [
        "Software Engineer", "Data Scientist", "Product Manager",
        "UX Designer", "DevOps Engineer", "Marketing Manager",
        "Sales Representative", "HR Manager", "Financial Analyst"
    ],
    
    # Seniority Levels
    "SENIORITY_LEVELS": ["Junior", "Mid", "Senior"],
    
    # Export Configuration
    "OUTPUT_FILE": "synthetic_data.csv"
}

print("Configuration loaded:")
print(f"  • Generating {CONFIG['NUM_ROWS']} rows")
print(f"  • Using LLM model: {CONFIG['LLM_MODEL']}")
print(f"  • Schema fields: {', '.join(CONFIG['SCHEMA'].keys())}")

## Step 3: Core Generator Class

The `SyntheticDataEngine` class orchestrates the entire data generation process:
- Uses **Faker** for standard fields (names, emails)
- Connects to **Ollama** for AI-generated content
- Implements **business logic** for data consistency
- Processes data in **batches** for efficiency

In [ ]:
class SyntheticDataEngine:
    """
    A comprehensive synthetic data generator that combines traditional fake data
    with AI-generated content using local LLMs via Ollama.
    """
    
    def __init__(self, config: Dict[str, Any]):
        """
        Initialize the data generator with configuration.
        
        Args:
            config: Configuration dictionary containing schema, LLM settings, etc.
        """
        self.config = config
        self.faker = Faker()
        Faker.seed(42)  # For reproducibility
        random.seed(42)
        
        print(f"🚀 SyntheticDataEngine initialized")
        print(f"   Model: {config['LLM_MODEL']}")
        
    def _generate_faker_field(self, field_name: str) -> str:
        """Generate data using Faker library."""
        if field_name == "Name":
            return self.faker.name()
        elif field_name == "Email":
            return self.faker.email()
        elif field_name == "Job Title":
            return random.choice(self.config["JOB_TITLES"])
        return ""
    
    def _apply_business_logic(self, row_data: Dict[str, Any]) -> Dict[str, Any]:
        """
        Apply business rules to ensure data consistency.
        
        Business Rules:
        1. Junior: 0-3 years of experience
        2. Mid: 3-7 years of experience
        3. Senior: 7+ years of experience
        """
        # Determine seniority level
        seniority = random.choice(self.config["SENIORITY_LEVELS"])
        row_data["Seniority"] = seniority
        
        # Apply experience rules based on seniority
        if seniority == "Junior":
            years_exp = random.randint(0, 3)
        elif seniority == "Mid":
            years_exp = random.randint(3, 7)
        else:  # Senior
            years_exp = random.randint(7, 15)
        
        row_data["Years of Experience"] = years_exp
        
        return row_data
    
    def _generate_ai_bio(self, job_title: str, seniority: str, years_exp: int) -> str:
        """
        Generate a contextual biography using Ollama LLM.
        
        Args:
            job_title: The person's job title
            seniority: Seniority level (Junior/Mid/Senior)
            years_exp: Years of experience
            
        Returns:
            AI-generated professional biography
        """
        prompt = f"""Write a brief professional bio (2-3 sentences) for a {seniority} {job_title} with {years_exp} years of experience. 
Make it realistic and professional. Focus on skills and achievements relevant to their role.
Do not include a name."""
        
        try:
            response = ollama.generate(
                model=self.config["LLM_MODEL"],
                prompt=prompt,
                options={
                    "temperature": self.config["LLM_TEMPERATURE"],
                    "num_predict": 100  # Limit response length
                }
            )
            return response['response'].strip()
        except Exception as e:
            # Fallback if LLM fails
            return f"Experienced {seniority} {job_title} with {years_exp} years in the industry."
    
    def generate_row(self) -> Dict[str, Any]:
        """
        Generate a single row of synthetic data.
        
        Returns:
            Dictionary containing all fields for one record
        """
        row_data = {}
        
        # Step 1: Generate Faker fields
        for field_name, field_type in self.config["SCHEMA"].items():
            if field_type == "faker":
                row_data[field_name] = self._generate_faker_field(field_name)
        
        # Step 2: Apply business logic for custom fields
        row_data = self._apply_business_logic(row_data)
        
        # Step 3: Generate AI content (LLM)
        if "AI Generated Bio" in self.config["SCHEMA"]:
            row_data["AI Generated Bio"] = self._generate_ai_bio(
                row_data["Job Title"],
                row_data["Seniority"],
                row_data["Years of Experience"]
            )
        
        return row_data
    
    def generate_batch(self, batch_size: int) -> List[Dict[str, Any]]:
        """
        Generate a batch of synthetic data rows.
        
        Args:
            batch_size: Number of rows to generate
            
        Returns:
            List of dictionaries, each representing a row
        """
        return [self.generate_row() for _ in range(batch_size)]
    
    def generate_dataset(self) -> pd.DataFrame:
        """
        Generate the complete dataset with progress tracking.
        
        Returns:
            Pandas DataFrame containing all synthetic data
        """
        num_rows = self.config["NUM_ROWS"]
        batch_size = self.config["BATCH_SIZE"]
        
        all_data = []
        num_batches = (num_rows + batch_size - 1) // batch_size
        
        print(f"\n📊 Generating {num_rows} rows in {num_batches} batches...")
        
        with tqdm(total=num_rows, desc="Generating data", unit="rows") as pbar:
            for batch_num in range(num_batches):
                # Calculate rows for this batch
                remaining_rows = num_rows - len(all_data)
                current_batch_size = min(batch_size, remaining_rows)
                
                # Generate batch
                batch_data = self.generate_batch(current_batch_size)
                all_data.extend(batch_data)
                
                # Update progress
                pbar.update(current_batch_size)
        
        # Convert to DataFrame
        df = pd.DataFrame(all_data)
        print(f"✓ Dataset generation complete! Shape: {df.shape}")
        
        return df

print("✓ SyntheticDataEngine class defined")

## Step 4: Generate the Dataset

Now we'll instantiate the engine and generate our synthetic data with a progress bar.

In [ ]:
# Initialize the engine
engine = SyntheticDataEngine(CONFIG)

# Generate the dataset
df = engine.generate_dataset()

## Step 5: Data Validation & Exploration

Let's examine the generated data to ensure quality and consistency.

In [ ]:
### 5.1 Preview First 5 Rows

print("📋 First 5 rows of generated data:\n")
df.head()

In [ ]:
### 5.2 Dataset Information

print("ℹ️  Dataset Information:\n")
print(f"Total Rows: {len(df)}")
print(f"Total Columns: {len(df.columns)}")
print(f"\nColumn Names and Types:")
print(df.dtypes)
print(f"\nMemory Usage: {df.memory_usage(deep=True).sum() / 1024:.2f} KB")

In [ ]:
### 5.3 Statistical Summary

print("📊 Statistical Summary:\n")
df.describe(include='all')

### 5.4 Business Logic Validation

Verify that our business rules are correctly applied:
- Junior employees should have 0-3 years of experience
- Mid-level employees should have 3-7 years
- Senior employees should have 7+ years

In [ ]:
print("🔍 Business Logic Validation:\n")

# Group by seniority and show experience statistics
validation = df.groupby('Seniority')['Years of Experience'].agg(['min', 'max', 'mean', 'count'])
print(validation)

# Check for violations
print("\n✓ Validation Results:")
junior_valid = df[df['Seniority'] == 'Junior']['Years of Experience'].max() <= 3
mid_valid = (df[df['Seniority'] == 'Mid']['Years of Experience'].min() >= 3) and \
            (df[df['Seniority'] == 'Mid']['Years of Experience'].max() <= 7)
senior_valid = df[df['Seniority'] == 'Senior']['Years of Experience'].min() >= 7

print(f"  Junior rules: {'✓ PASS' if junior_valid else '✗ FAIL'}")
print(f"  Mid rules: {'✓ PASS' if mid_valid else '✗ FAIL'}")
print(f"  Senior rules: {'✓ PASS' if senior_valid else '✗ FAIL'}")

### 5.5 Sample AI-Generated Bios

Let's examine a few AI-generated biographies to assess quality:

In [ ]:
print("🤖 Sample AI-Generated Biographies:\n")

# Show 3 random samples with different seniority levels
for seniority in ['Junior', 'Mid', 'Senior']:
    sample = df[df['Seniority'] == seniority].sample(1).iloc[0]
    print(f"{'='*70}")
    print(f"Name: {sample['Name']}")
    print(f"Title: {sample['Seniority']} {sample['Job Title']}")
    print(f"Experience: {sample['Years of Experience']} years")
    print(f"\nBio: {sample['AI Generated Bio']}")
    print()

## Step 6: Export Data

Save the generated dataset to a CSV file for further use.

In [ ]:
# Export to CSV
output_file = CONFIG["OUTPUT_FILE"]
df.to_csv(output_file, index=False)

print(f"✓ Dataset exported successfully!")
print(f"  File: {output_file}")
print(f"  Size: {len(df)} rows × {len(df.columns)} columns")

# Verify the file was created
import os
if os.path.exists(output_file):
    file_size = os.path.getsize(output_file) / 1024
    print(f"  File size: {file_size:.2f} KB")

## Summary & Key Takeaways

### What We Built
This notebook demonstrates a production-ready synthetic data generator that combines:
1. **Traditional fake data** (Faker) for structured fields
2. **AI-generated content** (Ollama) for contextual, realistic text
3. **Business logic** to ensure data consistency and realism
4. **Batch processing** for memory efficiency
5. **Progress tracking** for user feedback

### Architecture Highlights

#### 1. Modular Design
- Configuration separated from logic
- Reusable `SyntheticDataEngine` class
- Easy to extend with new fields or rules

#### 2. Data Quality
- Business rules enforce logical consistency
- Validation checks confirm rule compliance
- AI-generated content adds realism

#### 3. Scalability
- Batch processing prevents memory issues
- Can easily scale to thousands of rows
- Progress bars provide feedback for long operations

#### 4. Local-First Approach
- Uses Ollama for privacy and cost savings
- No API keys or external dependencies
- Full control over data generation

### Use Cases
- **Testing**: Generate realistic test data for applications
- **Training**: Create datasets for ML model training
- **Demos**: Populate demo environments with realistic data
- **Privacy**: Replace sensitive production data with synthetic alternatives

### Next Steps
To extend this generator, you could:
- Add more fields (phone numbers, addresses, departments)
- Implement more complex business rules
- Add data quality checks (email format validation, etc.)
- Generate related tables (employees → projects → tasks)
- Add visualization of generated data distributions
- Export to multiple formats (JSON, Parquet, SQL)

## Bonus: Quick Regeneration

Want to generate a different dataset? Just modify the CONFIG and re-run the generation cells!

In [ ]:
# Example: Generate a larger dataset with different settings
# Uncomment and run to try:

# CONFIG["NUM_ROWS"] = 100
# CONFIG["LLM_MODEL"] = "mistral"
# CONFIG["OUTPUT_FILE"] = "synthetic_data_large.csv"
# 
# engine_v2 = SyntheticDataEngine(CONFIG)
# df_v2 = engine_v2.generate_dataset()
# df_v2.to_csv(CONFIG["OUTPUT_FILE"], index=False)
# print(f"✓ New dataset created with {len(df_v2)} rows!")

print("💡 Tip: Uncomment the code above to generate a different dataset")